In [1]:
!git clone https://github.com/wanwuzhili/agent.git

Cloning into 'agent'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 12 (delta 0), reused 12 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 5.90 KiB | 5.90 MiB/s, done.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
cd /content/agent/

/content/agent


In [ ]:
!pip install -r requirements.txt

In [5]:
!cp /content/drive/MyDrive/data/eda_contest/Scan_User_Manual.pdf /content/agent/data/

# 1. LLM

In [6]:
import os

from langchain_openai import ChatOpenAI




model = ChatOpenAI(
    api_key="sk-e9915f8234cd4f1cb0f3c5eccdbd2860",
    base_url="https://api.deepseek.com",
    model="deepseek-flash",
    temperature=0,
)

# LLM + tools
# model_with_tools = model.bind_tools(tools)

In [7]:
response = model.invoke(
    [{
        "role": "user",
        "content": "hello, my name is Bob."
    }]
)

print(response.content)

Hello Bob, nice to meet you! How can I help you today?


# 2. Tools

- file
- shell
- rag
- (memory db)

In [8]:
# file tool
from langchain_community.agent_toolkits import FileManagementToolkit

file_toolkit = FileManagementToolkit(
    root_dir="./workspace",          # 限制工作区
    selected_tools=["read_file", "write_file"],
)

tools = {
    tool.name: tool
    for tool in file_toolkit.get_tools()
}

read_file = tools["read_file"]
write_file = tools["write_file"]

/tmp/ipykernel_604/69954654.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.agent_toolkits import FileManagementToolkit


In [9]:
write_file.invoke({
    "file_path": "test.py",
    "text": "print('hello')"
})

read_file.invoke({
    "file_path": "test.py"
})

"print('hello')"

In [10]:
# shell
from langchain_community.tools import ShellTool

shell = ShellTool(
    ask_human_input=False      # 请求
)

In [ ]:
result = shell.invoke({
    "commands": ["python workspace/test.py"]
})

print(result)

# 3. RAG and rag tool

- build_index.py: build vector database.
- retriever: use Chroma to create a retriever
- rag tool

*Embedding*: 一个外面训练好的模型，将自然语言转化成向量

*vector database*: 包含 Embedding 以及一些向量数据，所以可以将输入的语句转化成向量，但不知道输出哪些数据

*retriever*: 给 vector db 加上检索机制

*rag_tool*: 包装 retriever 给 agent 使用

```text
RAG/
├── data/
│   └── Scan_User_Manual.pdf
│
├── vector_db/
│
├── rag/
│   ├── __init__.py
│   └── rag.py
│
└── build_index.py
```

## Embedding

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

def get_embeddings():
    """
    Create the local Qwen3 Embedding model.
    """

    embeddings = HuggingFaceEmbeddings(
        model_name="Qwen/Qwen3-Embedding-0.6B",

        model_kwargs={
            "device": "cuda",
        },

        encode_kwargs={
            "normalize_embeddings": True,
        },

        query_encode_kwargs={
            "prompt": (
                "Instruct: Given an EDA technical question, "
                "retrieve relevant passages from technical documentation "
                "that can help answer the question.\n"
                "Query: "
            ),
            "normalize_embeddings": True,
        },
    )

    return embeddings

embeddings = get_embeddings()

## vector db

In [14]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

# 加载文件
PDF_PATH = "./data/Scan_User_Manual.pdf"

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

# chunk
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
)

chunks = splitter.split_documents(documents)

# 建立 vector db
VECTOR_DB_DIR = "./vector_db"
COLLECTION_NAME = "eda_knowledge"

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=VECTOR_DB_DIR,
)

## retriever

In [15]:

VECTOR_DB_DIR = "./vector_db"
COLLECTION_NAME = "eda_knowledge"

# 导入 vector db
vector_store = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=VECTOR_DB_DIR,
    )

retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": 4,
        },
    )

In [ ]:
query = "scan引擎的基本操作流程是什么？"
docs = retriever.invoke(query)

for i, doc in enumerate(docs, start=1):

    print("=" * 60)

    print(f"Document {i}")

    print("Source:")
    print(doc.metadata.get("source"))

    print("Page:")
    print(doc.metadata.get("page"))

    print("Content:")
    print(doc.page_content)

## rag tool

In [17]:
from langchain_core.tools import create_retriever_tool

rag_tool = create_retriever_tool(
        retriever=retriever,

        name="search_knowledge_base",

        description=(
            "Search the EDA technical knowledge base. "
            "Use this tool when the user asks about DFT, "
            "scan chain, wrapper scan, ScanInsertion, "
            "digital IC design, or other information that "
            "may be contained in the technical documents."
        ),
    )

In [ ]:
result = rag_tool.invoke({
    "query": "wrapper scan 和 boundary scan 有什么区别？"
})


print(result)

# 4. LangGraph: Agent loop

In [19]:
# state
from langgraph.graph import MessagesState

State = MessagesState

In [27]:
# node: llm_node, tool_node
from langchain.messages import HumanMessage, SystemMessage
from langgraph.prebuilt import ToolNode

tools = [read_file, write_file, shell, rag_tool]

model_with_tool = model.bind_tools(tools)

# llm_node
SYSTEM_PROMPT = """你是一个 Agent。可用工具有：
- read_file: 查看文件
- write_file: 写入文件
- shell: 操作终端
- rag_tool: 查询EDA scan相关的资料"""

def llm_node(state: State):
    response = model_with_tool.invoke(
        [
            SystemMessage(
                content=SYSTEM_PROMPT
            )
        ] + state["messages"]
    )

    return {
        "messages": [response]
    }

# tool node
tool_node = ToolNode(tools)

In [31]:
# build graph
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import tools_condition

builder = StateGraph(State)

builder.add_node("llm_node", llm_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "llm_node")

builder.add_conditional_edges(
    "llm_node",
    tools_condition,
    {
        "tools": "tools",
        END: END,
    },
)

builder.add_edge("tools", "llm_node")

graph = builder.compile()

In [22]:
def run_agent_stream():

    print("=" * 60)
    print("Agent System")
    print("Type 'exit' to quit.")
    print("=" * 60)

    while True:

        user_input = input("\nUser: ").strip()

        if not user_input:
            continue

        if user_input.lower() == "exit":
            break

        try:

            for update in graph.stream(
                {
                    "messages": [
                        {
                            "role": "user",
                            "content": user_input,
                        }
                    ]
                },
                stream_mode="updates",
            ):

                for node_name, state_update in update.items():

                    print(f"\n========== {node_name} ==========")

                    messages = state_update.get(
                        "messages",
                        []
                    )

                    for message in messages:

                        # AIMessage
                        if hasattr(message, "tool_calls"):

                            if message.tool_calls:
                                print("Tool Calls:")

                                for tool_call in message.tool_calls:
                                    print(
                                        f"  - {tool_call['name']}"
                                    )
                                    print(
                                        f"    args: {tool_call['args']}"
                                    )

                            elif message.content:
                                print(
                                    "Assistant:",
                                    message.content
                                )

                        # ToolMessage
                        elif message.type == "tool":
                            print("Tool Result:")
                            print(message.content)

        except Exception as e:

            print("\nAgent Error:")
            print(e)

In [33]:
run_agent_stream()

Agent System
Type 'exit' to quit.

User: scan 引擎的操作流程是什么？

========== llm_node ==========
Tool Calls:
  - search_knowledge_base
    args: {'query': 'scan 引擎 操作流程 scan engine operation flow'}
  - search_knowledge_base
    args: {'query': 'scan engine working principle steps'}

========== tools ==========
Tool Result:
Scan 引擎用户手册  Scan 引擎介绍 
 
 
2026-07-06  1 
1 Scan 引擎介绍 
1.1 概述 
1.2 工作流程 
1.3 支持的功能 
1.4 输入与输出 
1.5 调用 Scan 引擎 
1.6 约束和限制

Scan 引擎用户手册  运行 Scan 引擎 
 
2026-07-06  7 
2.1 运行 Scan 引擎的基本流程 
运行 Scan 引擎的基本流程适用于大多数设计。要运行 Scan 引擎，用户需要提供输入
文件以及定义运行 Scan 引擎过程中的各项设置。 
图 2-1 运行 Scan 引擎的基本流程 
 
运行 Scan 引擎的流程包括以下步骤： 
1. 读入文件 
读入综合后的网表文件、单元库文件和 CTL 文件。若已在综合阶段读入文件，则
可跳过该步骤。更多信息，请参考 2.1.1 读入文件章节。 
2. 设置 Scan 参数 
在读入文件后，需要指定设计的顶层。在指定顶层设计后，为了生成 Scan 
Chain，用户需要对插链参数进行一些设置，其中包括定义 DFT Signal、识别已
有插链信息、配置 Scan Chain、配置 Clock-Gating、配置 Wrapper Chain、配置
DFT Partition、配置 Scan Replacement。更多信息，请参考 2.1.2 设置 Scan 参
数章节。 
3. 执行 DRC 
在设置 Scan 参数以后，需要执行 DRC，执行 DRC 的过程中会触发 Design 
Learning。更多信息，请参考 2.1.3 

In [24]:
def run_agent_invoke():

    print("=" * 60)
    print("Agent System")
    print("Type 'exit' to quit.")
    print("=" * 60)

    while True:

        user_input = input("\nUser: ").strip()

        if not user_input:
            continue

        if user_input.lower() == "exit":
            print("Bye!")
            break

        try:

            result = graph.invoke(
                {
                    "messages": [
                        {
                            "role": "user",
                            "content": user_input,
                        }
                    ]
                }
            )

            # 最后一条消息通常就是 Agent 的最终回答
            last_message = result["messages"][-1]

            print("\nAssistant:")
            print(last_message.content)

        except Exception as e:

            print("\nAgent Error:")
            print(e)

In [ ]:
run_agent_invoke()